In [2]:
import pandas as pd


In [7]:
df = pd.read_excel("data/jobs_skill_matrix_updated_v41.xlsx")

In [8]:
df.shape

(65, 948)

In [11]:
df.columns[:50].tolist()

['company',
 'role',
 'seniority',
 'python',
 'tensorflow',
 'pytorch',
 'natural_language_processing',
 'large_language_models',
 'generative_artificial_intelligence',
 'vector_databases',
 'recommendation_systems',
 'model_deployment',
 'model_optimization',
 'computer_vision',
 'sql',
 'pandas',
 'jupyter',
 'c++',
 'rust',
 'elasticsearch',
 'information_retrieval',
 'distributed_computing',
 'google_cloud_platform',
 'financial_services',
 'stakeholder_management',
 'sales',
 'cloud_computing',
 'artificial_intelligence_strategy',
 'enterprise_technology',
 'digital_transformation',
 'consultative_selling',
 'presentation_skills',
 'relationship_building',
 'customer_engagement',
 'enterprise_artificial_intelligence_transformation',
 'r',
 'statistical_analysis',
 'optimization_models',
 'data_dashboards',
 'exploratory_data_analysis',
 'business_data_science',
 'marketing_analytics',
 'data_validation',
 'data_quality',
 'data_security',
 'business_strategy',
 'metrics_developme

In [12]:
len(df.columns)

948

In [15]:
basic_cols = ["company", "role", "seniority"]
skill_cols = [col for col in df.columns if col not in basic_cols]
df["interpersonal_skills_list"] = df[skill_cols].apply(
    lambda row: [skill for skill in skill_cols if row[skill]== 1],
    axis=1
)
df[["company", "role", "seniority","interpersonal_skills_list"]].head()

,company,role,seniority,interpersonal_skills_list
0,shopify,machine learning applied scientist,senior,"[python, tensorflow, pytorch, natural_language..."
1,shopify,machine learning engineer - search,senior,"[python, natural_language_processing, large_la..."
2,loblaw companies limited,"analyst, data operations",junior,"[python, sql, pandas, jupyter]"
3,google,"senior software engineer, artificial intellige...",senior,"[python, natural_language_processing, large_la..."
4,google,"senior software engineer, artificial intellige...",senior,"[python, natural_language_processing, large_la..."


# NLP Preparation

In [ ]:
df["job_text"] = (
    df["role"].fillna("")+ " "+
    df["seniority"].fillna("")+" "+
    df["interpersonal_skills_list"].apply(lambda x: " ".join(x))
)

/var/folders/s0/xx7dqt192j9572kb3v34spg00000gn/T/ipykernel_75703/2028597268.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["job_text"] = (


In [17]:
df[["company","role","job_text"]].head()

,company,role,job_text
0,shopify,machine learning applied scientist,machine learning applied scientist senior pyth...
1,shopify,machine learning engineer - search,machine learning engineer - search senior pyth...
2,loblaw companies limited,"analyst, data operations","analyst, data operations junior python sql pan..."
3,google,"senior software engineer, artificial intellige...","senior software engineer, artificial intellige..."
4,google,"senior software engineer, artificial intellige...","senior software engineer, artificial intellige..."


# Embeddings

In [18]:
from sentence_transformers import SentenceTransformer

In [19]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20641.85it/s]


# Create Embeddings For All 65 Jobs

In [20]:
job_embeddings = model.encode(
    df["job_text"].tolist(),
    show_progress_bar = True
)

Batches: 100%|██████████| 3/3 [00:02<00:00,  1.11it/s]


In [21]:
job_embeddings.shape

(65, 384)

In [22]:
sample_resume = """
Python
SQL
Machine Learning
Natural Language Processing
Large Language Models
PyTorch
TensorFlow
Data Analysis
"""

# Resume Matching

In [24]:
resume_embedding = model.encode(sample_resume)
resume_embedding

array([-3.47157829e-02, -1.15972258e-01,  6.04498340e-03,  3.21750268e-02,
        1.43479407e-02, -7.98009709e-03,  4.23936732e-02, -5.17938696e-02,
       -8.87848511e-02, -5.07160909e-02, -5.71954213e-02, -1.01672923e-02,
       -3.80304493e-02, -3.05101778e-02,  6.68516150e-03,  9.53115225e-02,
       -6.45304993e-02,  2.62999255e-02, -5.18986806e-02, -8.43846947e-02,
       -1.15898177e-02,  8.64691958e-02,  3.88677604e-03,  3.27908397e-02,
        2.94800252e-02, -8.95598670e-04,  1.76868699e-02, -1.89213138e-02,
        3.02911084e-02,  5.83901964e-02, -3.06122936e-02,  9.34301782e-03,
        1.88594554e-02,  1.30704880e-01,  1.23598706e-02, -6.28157286e-03,
       -8.90421718e-02, -3.68487053e-02, -1.49704041e-02, -4.52464931e-02,
        5.32513112e-02, -2.76948567e-02, -3.82213704e-02, -4.24597456e-05,
        8.61702934e-02, -2.61347517e-02, -2.47534197e-02, -4.11496349e-02,
        2.13045515e-02,  5.40651148e-03, -1.32383391e-01,  1.35365799e-02,
       -2.81776208e-02,  

In [25]:
resume_embedding.shape

(384,)

# Calculate Similarity

In [26]:
from sklearn.metrics.pairwise import cosine_similarity

In [28]:
similarities = cosine_similarity(
    [resume_embedding],
    job_embeddings
)

In [29]:
similarities.shape

(1, 65)

# Add Match Scores to DataFrame

In [30]:
df["match_score"]= similarities[0]
df[["company","role", "seniority" , "match_score"]].head()

/var/folders/s0/xx7dqt192j9572kb3v34spg00000gn/T/ipykernel_75703/696460036.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["match_score"]= similarities[0]


,company,role,seniority,match_score
0,shopify,machine learning applied scientist,senior,0.736840
1,shopify,machine learning engineer - search,senior,0.480247
2,loblaw companies limited,"analyst, data operations",junior,0.370092
3,google,"senior software engineer, artificial intellige...",senior,0.385682
4,google,"senior software engineer, artificial intellige...",senior,0.385682


# Top 10 matches

In [31]:
top_matches = (
    df[["company","role", "seniority" , "match_score"]]
    .sort_values("match_score", ascending= False)
)
top_matches.head(10)

,company,role,seniority,match_score
0,shopify,machine learning applied scientist,senior,0.736840
42,robertson & company ltd.,data scientist,mid,0.567003
19,insight global,data analyst,junior,0.561829
44,hays,machine learning engineer,mid,0.520127
61,hays,machine learning engineer,mid,0.517437
23,braze,"data scientist, artificial intelligence deploy...",mid,0.504185
32,scotiabank,"scientist - data, global artificial intelligen...",senior,0.490976
62,boardy,ai software engineer,mid,0.488879
51,qualcomm,ai/machine learning research engineer,senior,0.487576
33,ibm,machine learning developer intern,junior,0.484643


# Convert score to percentage

In [41]:

top_matches =  df[["company","role", "seniority" , "match_score"]].copy()
top_matches["match_percent"] = (
    top_matches["match_score"] * 100
).round(2)
top_matches["match_percent"] = top_matches["match_percent"].map(
    lambda x: f"{x:.2f}"
)

top_matches = top_matches.sort_values("match_score", ascending= False)
top_matches.head(10)

,company,role,seniority,match_score,match_percent
0,shopify,machine learning applied scientist,senior,0.736840,73.68
42,robertson & company ltd.,data scientist,mid,0.567003,56.70
19,insight global,data analyst,junior,0.561829,56.18
44,hays,machine learning engineer,mid,0.520127,52.01
61,hays,machine learning engineer,mid,0.517437,51.74
23,braze,"data scientist, artificial intelligence deploy...",mid,0.504185,50.42
32,scotiabank,"scientist - data, global artificial intelligen...",senior,0.490976,49.10
62,boardy,ai software engineer,mid,0.488879,48.89
51,qualcomm,ai/machine learning research engineer,senior,0.487576,48.76
33,ibm,machine learning developer intern,junior,0.484643,48.46


# Extract Test(PYPDF)

In [ ]:
from pypdf import PdfReader
reader = PdfReader("data/SAJEDEH KHALEDI_CV.pdf")
resume_text = ""

for page in reader.pages:
    resume_text += page.extract_text()

print(resume_text[:1000])

Ignoring wrong pointing object 14 0 (offset 0)


SAJEDEH KHALEDI Machine Learning / Data Scientist | Montreal, QC, Canada sajedeh.khaledi1@gmail.com | +1 438-462-9545 GitHub: github.com/Saja28 | LinkedIn: linkedin.com/in/sajedeh-khaledi   PROFESSIONAL SUMMARY Machine Learning–focused Data Scientist with a strong engineering background and a Data Science certificate from Le Wagon (December 2025). Experienced in building end-to-end machine learning systems, from data processing and model training to deployment and monitoring. Combines control engineering foundations with applied ML to deliver scalable, production-ready solutions.   TECHNICAL SKILLS Machine Learning & Data Science: Python, SQL, Pandas, NumPy, Scikit-learn, TensorFlow, Keras Modeling & Analytics: Feature Engineering, Model Evaluation, Statistical Analysis MLOps & Deployment: REST APIs, Streamlit, Google Cloud Platform, MLflow, Docker, Git Engineering Foundations: Control Systems, Signal Processing, FPGA / VHDL (foundational), Embedded C, AutoCAD  PROJECTS     NoDoze   AI

# Create Real Resume Embedding

In [46]:
resume_embedding.shape

(384,)

# Compare real CV with 65 jobs

In [49]:
similarities = cosine_similarity(
    [resume_embedding],
    job_embeddings
)

In [53]:
df["match_score"] = similarities[0]

top_matches = df[["company", "role", "seniority", "match_score"]].copy()

top_matches["match_percent"] = (
    top_matches["match_score"]*100
).round(2)

top_matches["match_percent"] = top_matches["match_percent"].map(
    lambda x: f"{x:.2f}"
)

top_matches = top_matches.sort_values("match_score", ascending= False)

top_matches.head(10)

,company,role,seniority,match_score,match_percent
0,shopify,machine learning applied scientist,senior,0.736840,73.68
42,robertson & company ltd.,data scientist,mid,0.567003,56.70
19,insight global,data analyst,junior,0.561829,56.18
44,hays,machine learning engineer,mid,0.520127,52.01
61,hays,machine learning engineer,mid,0.517437,51.74
23,braze,"data scientist, artificial intelligence deploy...",mid,0.504185,50.42
32,scotiabank,"scientist - data, global artificial intelligen...",senior,0.490976,49.10
62,boardy,ai software engineer,mid,0.488879,48.89
51,qualcomm,ai/machine learning research engineer,senior,0.487576,48.76
33,ibm,machine learning developer intern,junior,0.484643,48.46


# Recalculate similarity with real CV

In [58]:
real_resume_embedding = model.encode(resume_text)

real_similarities = cosine_similarity(
    [real_resume_embedding],
    job_embeddings
)

df["real_cv_match_score"] = real_similarities[0]

real_top_matches = df[
    ["company", "role", "seniority", "real_cv_match_score"]
].copy()


real_top_matches["match_percent"] = (
    real_top_matches["real_cv_match_score"]*100
).round(2)

real_top_matches["match_percent"] = real_top_matches["match_percent"].map(
    lambda x: f"{x:.2f}"
)

real_top_matches = real_top_matches.sort_values(
    "real_cv_match_score",
    ascending= False
)

real_top_matches.head(10)

,company,role,seniority,real_cv_match_score,match_percent
61,hays,machine learning engineer,mid,0.609506,60.95
44,hays,machine learning engineer,mid,0.601392,60.14
33,ibm,machine learning developer intern,junior,0.594658,59.47
36,rbc,data & ai engineer,mid,0.539686,53.97
32,scotiabank,"scientist - data, global artificial intelligen...",senior,0.513809,51.38
43,bmo,associate data scientist,mid,0.512719,51.27
14,jobright.ai,machine learning engineer - early career,junior,0.507308,50.73
38,canadian tire corporation,data scientist,mid,0.489869,48.99
57,apexon,senior data scientist,senior,0.486742,48.67
47,aviva canada,data scientist,mid,0.485646,48.56


# Missing Skills Analysis

In [61]:
best_match_index = real_top_matches.index[0]

best_job_skills = df.loc[best_match_index, "interpersonal_skills_list"]

best_job_skills[:20]

['python',
 'pytorch',
 'recommendation_systems',
 'model_deployment',
 'google_cloud_platform',
 'data_analysis',
 'communication',
 'machine_learning',
 'data_preparation',
 'data_cleaning',
 'remote_work',
 'collaboration',
 'testing',
 'feature_engineering',
 'model_evaluation',
 'airflow',
 'production_machine_learning',
 'kubernetes',
 'version_control',
 'model_training']

# Extract skills from your real CV

In [62]:
resume_text_lower = resume_text.lower()

resume_skills = [
    skill for skill in skill_cols
    if skill.replace("_", " ") in resume_text_lower
]

resume_skills[:30]

['python',
 'tensorflow',
 'sql',
 'pandas',
 'google_cloud_platform',
 'sales',
 'r',
 'statistical_analysis',
 'data_analytics',
 'forecasting',
 'machine_learning',
 'tpu',
 'data_science',
 'dashboards',
 'tableau',
 'demand_forecasting',
 'engineering',
 'analytics',
 'feature_engineering',
 'model_evaluation',
 'docker',
 'monitoring',
 'git',
 'keras',
 'model_training',
 'scala',
 'mlops',
 'github',
 'numpy',
 'healthcare']

# Find the missing skills automatically

In [63]:
missing_skills = [
    skill
    for skill in best_job_skills
    if skill not in resume_skills
]

missing_skills

['pytorch',
 'recommendation_systems',
 'model_deployment',
 'data_analysis',
 'communication',
 'data_preparation',
 'data_cleaning',
 'remote_work',
 'collaboration',
 'testing',
 'airflow',
 'production_machine_learning',
 'kubernetes',
 'version_control',
 'apache_spark',
 'documentation',
 'deep_learning',
 'google_bigquery',
 'software_engineering',
 'inference_pipelines',
 'training_pipelines',
 'search_ranking_models',
 'google_dataflow',
 'cloud_workflows',
 'training_validation',
 'overfitting',
 'generalization',
 'evaluation_pipelines']

# Keep only important missing technical skills

In [64]:
important_missing_skills = [
    skill for skill in missing_skills
    if skill in [
        "pytorch",
        "recommendation_systems",
        "model_deployment",
        "airflow",
        "kubernetes",
        "apache_spark",
        "deep_learning",
        "google_bigquery",
        "inference_pipelines",
        "training_pipelines",
        "search_ranking_models",
        "evaluation_pipelines"
    ]
]

important_missing_skills

['pytorch',
 'recommendation_systems',
 'model_deployment',
 'airflow',
 'kubernetes',
 'apache_spark',
 'deep_learning',
 'google_bigquery',
 'inference_pipelines',
 'training_pipelines',
 'search_ranking_models',
 'evaluation_pipelines']

In [65]:
matched_skills = [
    skill
    for skill in best_job_skills
    if skill in resume_skills
]

matched_skills

['python',
 'google_cloud_platform',
 'machine_learning',
 'feature_engineering',
 'model_evaluation',
 'model_training']

In [66]:
len(matched_skills)

6

In [67]:
best_match = real_top_matches.iloc[0]

match_report = {
    "company": best_match["company"],
    "role": best_match["role"],
    "seniority": best_match["seniority"],
    "match_percent": best_match["match_percent"],
    "matched_skills": matched_skills,
    "important_missing_skills": important_missing_skills
}

match_report

{'company': 'hays',
 'role': 'machine learning engineer',
 'seniority': 'mid',
 'match_percent': '60.95',
 'matched_skills': ['python',
  'google_cloud_platform',
  'machine_learning',
  'feature_engineering',
  'model_evaluation',
  'model_training'],
 'important_missing_skills': ['pytorch',
  'recommendation_systems',
  'model_deployment',
  'airflow',
  'kubernetes',
  'apache_spark',
  'deep_learning',
  'google_bigquery',
  'inference_pipelines',
  'training_pipelines',
  'search_ranking_models',
  'evaluation_pipelines']}

In [68]:
print(f"Company: {match_report['company']}")
print(f"Role: {match_report['role']}")
print(f"Seniority: {match_report['seniority']}")
print(f"Match Score: {match_report['match_percent']}%")

print("\nMatched Skills:")
for skill in match_report["matched_skills"]:
    print(f"✅ {skill}")

print("\nImportant Missing Skills:")
for skill in match_report["important_missing_skills"]:
    print(f"❌ {skill}")

Company: hays
Role: machine learning engineer
Seniority: mid
Match Score: 60.95%

Matched Skills:
✅ python
✅ google_cloud_platform
✅ machine_learning
✅ feature_engineering
✅ model_evaluation
✅ model_training

Important Missing Skills:
❌ pytorch
❌ recommendation_systems
❌ model_deployment
❌ airflow
❌ kubernetes
❌ apache_spark
❌ deep_learning
❌ google_bigquery
❌ inference_pipelines
❌ training_pipelines
❌ search_ranking_models
❌ evaluation_pipelines


I built an NLP-powered Resume Matcher using Sentence Transformers and cosine similarity that analyzes job descriptions, ranks opportunities against a real resume, identifies missing skills, and generates personalized career recommendations.

# Gemini Resume Recommendations